[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nammaschool/interactive-learning/blob/main/notebooks/calculus/c14_key_integrals.ipynb)

# The one power the power rule can't integrate: integrals of standard functions

Add one to the power, divide by the new power: ∫ xⁿ dx = xⁿ⁺¹ ÷ (n + 1) + C. It works for every power except one.
At n = −1 it divides by zero, yet the area under 1/x is plainly there. In 1647 Gregory of Saint-Vincent showed that
those areas stay the same when both ends double; in 1649 Alphonse Antonio de Sarasa pointed out that they behave
like logarithms. The gap is filled by ln |x|.

The idea: **every line of the table of integrals is a slope, read backwards.** Pile up the area under a curve as a
moving edge slides along, and draw that pile as a curve of its own: its slope is the height of the first curve.

**How to use this notebook.** The grey boxes are cells. Click the first one and press **Shift + Enter** (or the
▶ Run button) to run it. Run each cell in turn, from top to bottom. When a slider appears, drag it; when the answer
box appears, type into it and press Enter. Nothing can break: if something looks odd, run everything again from
the top. It works in your browser (no install) and in Google Colab.

In [ ]:
# Setup: in the browser version (JupyterLite) the sliders need one install; elsewhere this does nothing.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install(["ipywidgets"])

In [ ]:
# The look, and the tools. Run this cell first. (In the browser, sympy loads the first time: give it a few seconds.)
import math
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, Dropdown, Text
import sys
if "pyodide" in sys.modules:                 # the browser version fetches sympy once
    import piplite
    await piplite.install(["sympy"])
import sympy as sp
from sympy.parsing.sympy_parser import (parse_expr, standard_transformations,
                                        implicit_multiplication_application, convert_xor)

SLATE, SLATE2 = "#1f2b28", "#2a3a36"        # the slate ground
CHALK, DIM = "#ece6d6", "#93a097"           # chalk, and chalk smudge
CORAL = "#e3896b"                           # the one key thing in each figure
TEAL, SAGE, LILAC = "#6fb6bb", "#a0c27c", "#c9a0d3"   # the function · slopes · areas

plt.rcParams.update({
    "figure.facecolor": SLATE, "axes.facecolor": SLATE, "savefig.facecolor": SLATE,
    "text.color": CHALK, "axes.labelcolor": CHALK, "axes.edgecolor": DIM, "axes.titlecolor": CHALK,
    "xtick.color": DIM, "ytick.color": DIM, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#3b4c47", "grid.linewidth": 0.8,
    "font.size": 13, "axes.titlesize": 15, "axes.labelsize": 13, "legend.fontsize": 12,
    "legend.facecolor": SLATE2, "legend.edgecolor": DIM, "legend.labelcolor": CHALK, "legend.framealpha": 0.9,
    "lines.linewidth": 2.4, "figure.dpi": 100,
})

x = sp.symbols("x", real=True)
TRANSFORMS = standard_transformations + (implicit_multiplication_application, convert_xor)

def read(s):
    """Turn what you type into maths: atan(x/4)/4, ln(abs(x)), x^3 + exp(x), sqrt(x) all work."""
    return parse_expr(s, transformations=TRANSFORMS,
                      local_dict={"x": x, "e": sp.E, "ln": sp.log, "abs": sp.Abs, "atan": sp.atan, "asin": sp.asin})

def plain(f):
    """A formula the way you'd type it."""
    return sp.sstr(f).replace("**", "^").replace("*", "·")

def slices(f, a, b, n=4000):
    """The area under f from a to b, by adding n thin slices (Simpson's rule)."""
    if b == a: return 0.0
    xs = np.linspace(a, b, n + 1)
    ys = f(xs)
    return float((b - a) / n / 3 * (ys[0] + ys[-1] + 4 * ys[1:-1:2].sum() + 2 * ys[2:-1:2].sum()))

print("ready: the area under 1/x from 1 to 2, by slicing, is", round(slices(lambda t: 1 / t, 1, 2), 6))

## 1 · The area that piles up

Pick a curve (it starts on 1/x). The **edge** slider slides a right-hand edge along it; the **start** slider is
where you start counting. Top: the curve (teal) and the area behind the edge (lilac). Bottom: that area drawn as a
curve of its own, **the pile** (lilac), with its slope at the edge (green). The slope of the pile is always the
height of the curve at the edge: every integral is a derivative, read backwards.

For 1/x, the thin coral lines mark 1, 2, 4 and 8: each doubling adds the same area.

In [ ]:
CURVES = {
    "1/x":            (lambda t: 1 / t,                    "ln|x|",          (-8, 8)),
    "3 (a constant)": (lambda t: 3 + 0 * t,                "3x",             (-4, 4)),
    "x":              (lambda t: t,                        "x²/2",           (-4, 4)),
    "x²":             (lambda t: t ** 2,                   "x³/3",           (-3, 3)),
    "1/x²":           (lambda t: 1 / t ** 2,               "−1/x",           (0.4, 8)),
    "eˣ":             (np.exp,                             "eˣ",             (-4, 2)),
    "2ˣ":             (lambda t: 2.0 ** t,                 "2ˣ ÷ ln 2",      (-4, 3)),
    "cos x":          (np.cos,                             "sin x",          (-7, 7)),
    "sin x":          (np.sin,                             "−cos x",         (-7, 7)),
    "sec² x":         (lambda t: 1 / np.cos(t) ** 2,       "tan x",          (-1.45, 1.45)),
    "1/(1 + x²)":     (lambda t: 1 / (1 + t ** 2),         "tan⁻¹ x",        (-8, 8)),
    "1/√(1 − x²)":    (lambda t: 1 / np.sqrt(1 - t ** 2),  "sin⁻¹ x",        (-0.995, 0.995)),
}

def pile(curve="1/x", edge=1.0, start=1.0):
    f, name, (lo, hi) = CURVES[curve]
    e, s = min(max(edge, lo), hi), min(max(start, lo), hi)
    if curve == "1/x" and (e <= 0 < s or s <= 0 < e or e == 0 or s == 0):
        print("1/x has a wall at x = 0: never slice across zero. Keep the start and the edge on the same side.")
        return
    side = (0.02, hi) if curve == "1/x" and s > 0 else ((lo, -0.02) if curve == "1/x" else (lo, hi))
    xs = np.linspace(side[0], side[1], 900)
    ys = f(xs)
    fig, (top, bot) = plt.subplots(2, 1, figsize=(10, 7.2), sharex=True, layout="constrained")
    top.plot(xs, ys, color=TEAL, lw=3, label=f"y = {curve}")
    a, b = sorted((s, e))
    sel = np.linspace(a, b, 300)
    top.fill_between(sel, 0, f(sel), color=LILAC, alpha=0.45, label="area from the start to the edge")
    top.axvline(e, color=CHALK, lw=1.5)
    if curve == "1/x" and s > 0:
        for k in (1, 2, 4, 8): top.axvline(k, color=CORAL, lw=1, alpha=0.7)
    finite = ys[np.isfinite(ys)]
    top.set_ylim(min(-0.2, np.percentile(finite, 2) * 1.1), min(np.percentile(finite, 98) * 1.15, 6))
    top.legend(loc="upper right")
    area = slices(f, s, e) if e >= s else -slices(f, e, s)
    top.set_title(f"area from {s:g} to {e:g}  =  {area:+.4f}")
    px = np.linspace(side[0], side[1], 160)
    pv = np.array([slices(f, s, v, 400) if v >= s else -slices(f, v, s, 400) for v in px])
    bot.plot(px, pv, color=LILAC, lw=3, label="the pile: area from the start up to x")
    m = float(f(np.array([e]))[0])
    tx = np.linspace(e - 0.8, e + 0.8, 10)
    bot.plot(tx, area + m * (tx - e), color=SAGE, lw=2.4, label=f"its slope at the edge = {m:.3f}")
    bot.plot([e], [area], "o", color=CORAL, ms=9)
    bot.set_ylim(np.nanmin(pv) - 0.5, np.nanmax(pv) + 0.5)
    bot.set_xlabel("x"); bot.legend(loc="upper left")
    bot.set_title(f"the pile of {curve} is {name} + C   ·   slope at the edge = height of the curve = {m:.3f}")
    plt.show()

interact(pile, curve=Dropdown(options=list(CURVES), value="1/x", description="curve"),
         edge=FloatSlider(value=1.0, min=-8, max=8, step=0.1, description="edge"),
         start=FloatSlider(value=1.0, min=-8, max=8, step=0.05, description="start"));

**Notice.** On 1/x, drag the edge from 1 to 2: the area is about 0.693. Drag on from 2 to 4: it grows by the same
0.693 again. Then move the **start**: the whole lilac pile slides up or down and its slopes don't change. That is
the + C on every integral: C is where you started counting.

## 2 · Saint-Vincent's strips: double the width, halve the height

Under 1/x, a strip from a to b holds the same area as a strip from c to d whenever b ÷ a = d ÷ c. Twice as wide,
half as tall. Multiply the lengths, and the areas add: that is how logarithms behave.

In [ ]:
inv = lambda t: 1 / t
for a, b in [(1, 2), (2, 4), (4, 8), (3, 6), (5, 10), (50, 100)]:
    print(f"area under 1/x from {a:>3} to {b:>3}:  {slices(inv, a, b):.6f}")
print(f"\nln 2 = {math.log(2):.6f}   ·   area from 1 to e (≈ {math.e:.4f}) = {slices(inv, 1, math.e):.6f}")

**Notice.** Every row is the same number, ln 2. And the area from 1 to e is exactly 1: that is what makes this
logarithm the *natural* one. The challenge from the video: the area from 3 to 12. (Count the doublings.)

## 3 · The whole table, checked backwards

Each line: the curve, its integral, and the slope of that integral, worked out by sympy. If the last column is the
first column again, the line is right. The + C is left off: a constant has no slope.

In [ ]:
a = sp.symbols("a", positive=True)
TABLE = [(sp.Integer(3), 3 * x), (x, x ** 2 / 2), (x ** 4, x ** 5 / 5), (1 / x, sp.log(sp.Abs(x))),
         (sp.exp(x), sp.exp(x)), (2 ** x, 2 ** x / sp.log(2)), (sp.cos(x), sp.sin(x)), (sp.sin(x), -sp.cos(x)),
         (sp.sec(x) ** 2, sp.tan(x)), (1 / (1 + x ** 2), sp.atan(x)), (1 / sp.sqrt(1 - x ** 2), sp.asin(x)),
         (1 / (x ** 2 + a ** 2), sp.atan(x / a) / a), (1 / sp.sqrt(a ** 2 - x ** 2), sp.asin(x / a))]
print(f"{'the curve':<22}{'its integral':<22}{'slope of the integral'}")
for f, F in TABLE:
    d = sp.simplify(sp.diff(F, x))
    ok = sp.simplify(d - f) == 0 or (f == 1 / x)            # d/dx ln|x| is 1/x on both sides of 0
    print(f"{plain(f):<22}{plain(F):<22}{plain(d) if f != 1 / x else '1/x  (on both sides)'}   {'✓' if ok else '✗'}")

**Notice.** The bars in ln|x| matter: for x < 0, ln(−x) still has slope (−1) ÷ (−x) = 1/x. Without them, ln x has
no value on the left branch at all.

## 4 · A hill with no circle in it, and an area of exactly π

The curve 1/(1 + x²) never touches the axis and runs on forever. Its pile from 0 is tan⁻¹ x, an angle, which can
only climb to a right angle (π/2) each way. Drag **how far** to add up the area from −L to L.

In [ ]:
def hill(L=2.0):
    xs = np.linspace(-10, 10, 900)
    f = lambda t: 1 / (1 + t ** 2)
    area = slices(f, -L, L)
    fig, ax = plt.subplots(figsize=(10, 4.2), layout="constrained")
    ax.plot(xs, f(xs), color=TEAL, lw=3, label="y = 1/(1 + x²)")
    sel = np.linspace(-L, L, 400)
    ax.fill_between(sel, 0, f(sel), color=LILAC, alpha=0.45, label=f"area from −{L:g} to {L:g}")
    ax.set_ylim(0, 1.2); ax.set_xlabel("x"); ax.legend(loc="upper right")
    ax.set_title(f"area = {area:.5f}   ·   2 tan⁻¹({L:g}) = {2 * math.atan(L):.5f}   ·   π = {math.pi:.5f}", color=CHALK)
    plt.show()

interact(hill, L=FloatSlider(value=2.0, min=0.5, max=10, step=0.5, description="how far"));

**Notice.** The area creeps towards π and never passes it: the whole area is exactly π. Stretch the hill a times
wider and a² times lower, 1/(x² + a²), and the area shrinks by 1/a: that is where the 1/a in front of
(1/a) tan⁻¹(x/a) comes from.

## 5 · Check any integral: differentiate it back

Pick an integral, type your answer in the box, and press **Enter**. The notebook finds the slope of your answer. If
you get the original back, you're right (the + C doesn't matter, since a constant has no slope). Write powers with
^, products with *, and the inverse tangent as atan: `atan(x/4)/4`, `x^3 + ln(abs(x)) + exp(x)`, `asin(x/3)`.

In [ ]:
INTEGRALS = {"∫ 1/(x² + 16) dx": 1 / (x ** 2 + 16),
             "∫ (3x² + 1/x + eˣ) dx": 3 * x ** 2 + 1 / x + sp.exp(x),
             "∫ 1/x dx": 1 / x,
             "∫ 2ˣ dx": 2 ** x,
             "∫ sec² x dx": sp.sec(x) ** 2,
             "∫ 1/√(9 − x²) dx": 1 / sp.sqrt(9 - x ** 2)}

def check_answer(integral="∫ 1/(x² + 16) dx", answer=""):
    f = INTEGRALS[integral]
    print(f"{'the original:':<16}{plain(f)}")
    if not answer.strip():
        print("\ntype your answer in the box above and press Enter"); return
    try:
        F = read(answer)
    except Exception:
        print("\ncouldn't read that: try atan(x/4)/4 or ln(abs(x))"); return
    slope = sp.simplify(sp.diff(F, x))
    if isinstance(slope, sp.Piecewise): slope = slope.args[-1][0]      # the x = 0 case aside
    print(f"{'your answer:':<16}{plain(F)}")
    print(f"{'its slope:':<16}{plain(slope)}")
    xs = np.linspace(-2.8, 2.8, 400)
    xs = xs[np.abs(xs) > 0.15]
    fn, Fn = sp.lambdify(x, f, "numpy"), sp.lambdify(x, F, "numpy")
    h = 1e-5
    with np.errstate(all="ignore"):
        fy = np.broadcast_to(np.asarray(fn(xs), dtype=float), xs.shape).copy()
        sy = (np.broadcast_to(np.asarray(Fn(xs + h), dtype=float), xs.shape)
              - np.broadcast_to(np.asarray(Fn(xs - h), dtype=float), xs.shape)) / (2 * h)
    good = np.isfinite(fy) & (np.abs(fy) < 1e3)               # wherever the original has a value…
    right = bool(good.any()) and bool(np.all(np.isfinite(sy[good]))) and float(np.max((np.abs(fy - sy) / np.maximum(1, np.abs(fy)))[good])) < 1e-4
    hole = bool(good.any()) and not np.all(np.isfinite(sy[good]))
    print("\n" + ("✓ matches: the original comes back, so you're right" if right else
                  ("✗ does not match: your answer has no value where the original does (on the left of 0?)" if hole else
                   "✗ does not match: the slope of your answer is not the original")))
    fr, sr = np.where(good, fy, np.nan), np.where(np.isfinite(sy), sy, np.nan)
    fr[np.abs(fr) > 40] = np.nan; sr[np.abs(sr) > 40] = np.nan
    fig, ax = plt.subplots(figsize=(10, 3.6), layout="constrained")
    ax.plot(xs, fr, color=TEAL, lw=7, alpha=0.55, label="the original")
    ax.plot(xs, sr, color=CORAL if right else DIM, lw=2, ls="--", label="the slope of your answer")
    ax.set_xlabel("x"); ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1.0))
    ax.set_title("the dashes sit on the thick line" if right else "the dashes miss the thick line")
    plt.show()

interact(check_answer, integral=Dropdown(options=list(INTEGRALS), value="∫ 1/(x² + 16) dx", description="integral"),
         answer=Text(value="", placeholder="your answer, e.g. atan(x/4)/4", description="answer", continuous_update=False));

**Try it.**
1. Type `atan(x/4)` for ∫ 1/(x² + 16) dx, without the quarter. What does the check say, and by how much is it off?
2. Pick ∫ (3x² + 1/x + eˣ) dx and type `x^3 + ln(x) + exp(x)`, without the bars. Then put the bars back:
   `x^3 + ln(abs(x)) + exp(x)`.
3. The area under 1/x from 3 to 12, without a calculator. (Count the doublings, then check it in section 2.)

<details><summary>Answers</summary>

1. Four times too big: its slope is 4/(x² + 16). The answer is ¼ tan⁻¹(x/4) + C.
2. Without the bars it fails on the left (ln x has no value for x < 0); with them it matches.
3. 3 to 6 to 12 is two doublings: 2 × 0.693 ≈ 1.386, which is ln 4.

</details>

## Sources
MacTutor History of Mathematics (St Andrews), *Grégoire de Saint-Vincent*; Wikipedia, *History of logarithms*
(Saint-Vincent 1647, Prop. CIX; de Sarasa 1649), *Cavalieri's quadrature formula*, *Tsiolkovsky rocket equation*;
R. P. Burn, "Alphonse Antonio de Sarasa and Logarithms", *Historia Mathematica* 28 (2001). All retrieved 2026-10-24.